In [ ]:
%pip install pandas
%pip install dash

In [59]:
import pandas as pd

In [60]:
#Loading raw data into a dataframe
df = pd.read_csv('https://raw.githubusercontent.com/pawanshukla9/data/refs/heads/main/ecommerce_customer_churn_dataset.csv')
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


In [31]:
# Remove invalid ages
df = df[df["Age"].isna() | df["Age"].between(16, 100)]

# Fix values that should not be negative or above 100
df["Total_Purchases"] = df["Total_Purchases"].clip(lower=0)
df["Cart_Abandonment_Rate"] = df["Cart_Abandonment_Rate"].clip(0, 100)
df["Returns_Rate"] = df["Returns_Rate"].clip(0, 100)
df["Discount_Usage_Rate"] = df["Discount_Usage_Rate"].clip(0, 100)

# Fill missing values
df["Wishlist_Items"] = df["Wishlist_Items"].fillna(0)

# Fill missing numeric column values with median
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Convert selected columns to integers
int_cols = [
    "Age",
    "Customer_Service_Calls",
    "Product_Reviews_Written",
    "Wishlist_Items",
    "Payment_Method_Diversity",
    "Days_Since_Last_Purchase",
    "Login_Frequency"
]

df[int_cols] = df[int_cols].round().astype(int)

# Round remaining float columns to 2 decimal places
float_cols = df.select_dtypes(include="float").columns
df[float_cols] = df[float_cols].round(2)

# Clean text columns
text_cols = ["Gender", "Country", "City", "Signup_Quarter"]

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# Fix country abbreviations
df["Country"] = df["Country"].replace({
    "Uk": "UK",
    "Usa": "USA"
})

# Remove duplicate rows
df = df.drop_duplicates()

# Create useful dashboard columns
df["Churn_Status"] = df["Churned"].map({
    0: "Active",
    1: "Churned"
})

df["Payment_Mode"] = df["Payment_Method_Diversity"].map({
    1: "Cash",
    2: "Debit Card",
    3: "Credit Card",
    4: "PayPal",
    5: "Others"
})

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[16, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "55 and above"]
)

df["Engagement_Tier"] = pd.cut(
    df["Login_Frequency"],
    bins=[-1, 4, 12, 24, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

df["Value_Segment"] = pd.qcut(
    df["Lifetime_Value"],
    q=4,
    labels=["Bronze", "Silver", "Gold", "Platinum"]
)

# Save cleaned file
df.to_csv("e-commerce_churn_clean.csv", index=False)

In [61]:
# Loading clean data into a dataframe
df = pd.read_csv('https://raw.githubusercontent.com/pawanshukla9/data/refs/heads/main/e-commerce_churn_clean.csv')
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter,Churn_Status,Payment_Mode,Age_Group,Engagement_Tier,Value_Segment
0,43,Male,France,Marseille,2.9,14,27.4,6.0,50.6,3,...,1,953.33,2278.0,0,Q1,Active,Cash,36-45,High,Silver
1,36,Male,UK,Manchester,1.6,15,42.7,10.3,37.7,1,...,3,1067.47,3028.0,0,Q4,Active,Credit Card,36-45,High,Silver
2,45,Female,Canada,Vancouver,2.9,10,24.8,1.6,70.9,1,...,2,1289.75,2317.0,0,Q4,Active,Debit Card,36-45,Medium,Gold
3,56,Female,USA,New York,2.6,10,38.4,14.8,41.7,9,...,3,2340.92,2674.0,0,Q1,Active,Credit Card,55 and above,Medium,Platinum
4,35,Male,India,Delhi,3.1,29,51.4,8.4,19.1,9,...,4,3041.29,5354.0,0,Q4,Active,PayPal,26-35,Very High,Platinum


In [ ]:
# START FROM BELOW. DO NOT CHANGE THE CODE ABOVE. 